# Banking Fraud & Risk Analytics Platform

# 06. Feature Engineering

## Objective

The objective of feature engineering is to transform raw transaction data into meaningful business-oriented features that enhance data analysis, improve reporting, and provide deeper insights into transaction behavior. These engineered features simplify business analysis, support interactive dashboards, and prepare the dataset for future predictive modeling without altering the original transactional information.

---



### Importing All Required Libraries

In [1]:
import os
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

In [2]:
transactions = pd.read_csv("../data/processed/cleaned_transaction.csv")
accounts = pd.read_csv("../data/processed/cleaned_accounts.csv")

In [4]:

print("Transactions Shape :", transactions.shape)
print("Accounts Shape     :", accounts.shape)
print("="*60)

transactions.head()

Transactions Shape : (5078336, 11)
Accounts Shape     : (518581, 5)


,timestamp,from_bank,sender_account,to_bank,receiver_account,amount_received,receiving_currency,amount_paid,payment_currency,payment_format,is_laundering
0,2022-09-01 00:20:00,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022-09-01 00:20:00,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022-09-01 00:00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022-09-01 00:02:00,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022-09-01 00:06:00,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


In [6]:
#creating copy of dataset
transaction_feature=transactions.copy()
accounts_feature=accounts.copy()

In [13]:
#Convert Timestamp to Datetime
transaction_feature["timestamp"] = pd.to_datetime(transaction_feature["timestamp"])

In [15]:
#Transaction Hour
transaction_feature["transaction_hour"] = transaction_feature["timestamp"].dt.hour

In [23]:
#Day of week
transaction_feature["day_of_week"] = transaction_feature["timestamp"].dt.day_name()

In [26]:
#Month of transaction
transaction_feature["transaction_month"] = transaction_feature["timestamp"].dt.month_name()

In [27]:
transaction_feature.sample(4)

,timestamp,from_bank,sender_account,to_bank,receiver_account,amount_received,receiving_currency,amount_paid,payment_currency,payment_format,is_laundering,transaction_hour,day_of_week,transaction_month
2901201,2022-09-06 06:23:00,11107,801BE4DF0,218438,8075B8E50,2823.39,US Dollar,2823.39,US Dollar,Credit Card,0,6,Tuesday,September
3892459,2022-09-08 07:04:00,245328,810BA6110,217,81125F900,7.62,Shekel,7.62,Shekel,Cheque,0,7,Thursday,September
4908050,2022-09-10 02:33:00,122055,8090EBA70,8,8095B77A0,36986.62,Canadian Dollar,36986.62,Canadian Dollar,Cheque,0,2,Saturday,September
2715490,2022-09-05 21:09:00,29992,80C2E9EA0,34828,80E942990,112334.39,Yuan,112334.39,Yuan,Cash,0,21,Monday,September


In [28]:
#Amount Category
transaction_feature["amount_category"] = pd.qcut(
    transaction_feature["amount_paid"],
    q=3,
    labels=["Low", "Medium", "High"]
)

In [29]:
transaction_feature.sample(4)

,timestamp,from_bank,sender_account,to_bank,receiver_account,amount_received,receiving_currency,amount_paid,payment_currency,payment_format,is_laundering,transaction_hour,day_of_week,transaction_month,amount_category
4314935,2022-09-09 03:08:00,17425,80485DC80,244684,8143B1220,33.75,US Dollar,33.75,US Dollar,Cheque,0,3,Friday,September,Low
1110240,2022-09-01 23:35:00,15895,805C82CB0,15895,805C82CB0,2809228.84,Rupee,2809228.84,Rupee,Reinvestment,0,23,Thursday,September,High
3563409,2022-09-07 15:11:00,21,8039F1600,15,803A48F00,26149.23,Yen,26149.23,Yen,Cash,0,15,Wednesday,September,High
898373,2022-09-01 17:35:00,7042,803A41550,7042,803A41550,145165.01,US Dollar,145165.01,US Dollar,Reinvestment,0,17,Thursday,September,High


In [30]:
#Same Bank Transfer
transaction_feature["same_bank_transfer"] = (
    transaction_feature["from_bank"] ==
    transaction_feature["to_bank"]
)

In [32]:
#Same Currency Transfer
transaction_feature["same_currency_transaction"] = (
    transaction_feature["payment_currency"] ==
    transaction_feature["receiving_currency"]
)

#### Engineered Feature

In [34]:
transaction_feature[
    [
        "timestamp",
        "transaction_hour",
        "day_of_week",
        "transaction_month",
        "amount_paid",
        "amount_category",
        "same_bank_transfer",
        "same_currency_transaction"
    ]
].head()

,timestamp,transaction_hour,day_of_week,transaction_month,amount_paid,amount_category,same_bank_transfer,same_currency_transaction
0,2022-09-01 00:20:00,0,Thursday,September,3697.34,Medium,True,True
1,2022-09-01 00:20:00,0,Thursday,September,0.01,Low,False,True
2,2022-09-01 00:00:00,0,Thursday,September,14675.57,High,True,True
3,2022-09-01 00:02:00,0,Thursday,September,2806.97,Medium,True,True
4,2022-09-01 00:06:00,0,Thursday,September,36682.97,High,True,True


In [35]:
#Saving data engineered data
os.makedirs("../data/processed", exist_ok=True)

transaction_feature.to_csv(
    "../data/processed/feature_engineered_transactions.csv",
    index=False
)


---
## Features Engineered

In this section, the following business-driven features has been created:

- **Transaction Hour** – Extracts the hour from the transaction timestamp to analyze hourly transaction patterns.
- **Day of Week** – Identifies the day on which the transaction occurred for weekly trend analysis.
- **Transaction Month** – Extracts the month to support monthly reporting and seasonal analysis.
- **Amount Category** – Groups transaction amounts into Low, Medium, and High categories for easier business interpretation.
- **Same Bank Transfer** – Indicates whether the sender and receiver belong to the same bank.
- **Same Currency Transaction** – Identifies whether the payment and receiving currencies are identical or involve currency conversion.